In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(".."))

from competition import solve_adjustable_model

## Investigating the effects of competition among biobased products within the bioeconomy matrix - `solve_adjustable_model` function documentation

The `solve_adjustable_model` function is a highly flexible simulation tool designed to analyze how the bioeconomy's interconnected production system adjusts to changing demand.

### Core concept

The function takes an LCI-like matrix representing the bioeconomy, a set of new demand targets for specific biobased products, and a collection of physical rules and modeling penalties. It then calculates a new, adjusted matrix that meets these demands while maintaining the structural integrity of the bioeconomy.

This is particularly useful for answering "What if?" scenarios:
> *"If the demand for bioenergy increases by 20%, which products competing for the same biomass as feedstock will be unable to meet their current demand, and how much demand will not be met?"*

The model works by defining a complex optimization problem. It tries to satisfy the requested new demand values for the targeted biobased product while penalizing different types of changes (e.g., creating a deficit of another product). The final result is the lowest-penalty solution. Surpluses may also be generated when the demand results in excess output in other activities.

The optimized matrix allows to identify and quantify the deficit linked with the requested new demand. This is the main output of the model.

#### How it works
1.  **Network mapping:** the model starts from the `target_rows` (the products whose demand is changing) and performs a graph traversal to identify all upstream and downstream sectors that are physically linked to these products.
2.  **Optimization:** it defines a linear programming problem over this identified sub-network.
3.  **Resolution:** it calculates the optimal adjustment that satisfies the `new_demands` while minimizing "penalties" for undesirable changes, such as creating a supply **deficit** or generating an unmanageable **surplus**.

### Key parameters

#### 1. Basic inputs
*   **`M_csv_path`** (string | DataFrame): the file path to your input matrix (CSV) or a pandas DataFrame.
*   **`row_class`** (dict): a dictionary classifying every row index into one of three categories:
    *   `'primary'`: raw biomass entering the economy (e.g., crops, forestry, imports).
    *   `'intermediate'`: biomass flows requiring further processing within the matrix.
    *   `'output'`: final products leaving the bioeconomy or used by external industries.
*   **`target_rows`** (list of ints): the integer indexes of the rows where demand is changing.
*   **`new_demands`** (list of floats): the new target values for the total row sums of the `target_rows`.
*   **`row_ratios` / `column_ratios`** (dict): dictionaries enforcing fixed proportions. Use this to lock "recipes" (column ratios) or co-production rates (row ratios) so they don't change during the simulation.
*   **`slack_rows`** (list, Default: `None`): Reserved for internal handling of slack constraints.
*   **`exclude_deficit_rows`** (list, Default: `None`): Row indices that must not experience production cuts.
*   **`exclude_surplus_rows`** (list, Default: `None`): Row indices that must not experience forced overproduction.
*   **`eps`** (float, Default: `1e-9`): Small numerical constant used for non-zero constraints.
*   **`distribution`** (string, Default: `'row_sum'`): set to `'row_sum'` to enforce edge proportionality: any necessary deficits are distributed proportionally among eligible output sectors based on their size, rather than falling arbitrarily on a single sector.
*   **Note on "Eligible Output":** a row is "eligible" to share a deficit only if it is: (1) classified as `'output'`, (2) connected to the target network mapped, and (3) not listed in `exclude_deficit_rows` (see below).
*   **`special_rows`** (list of ints, Default: `None`): A list of integer row indices. These rows are **exempt** from the edge proportionality  (when `distribution='row_sum'`). Their deficit is free to vary independently to satisfy the solver's objective, while other output rows must share the burden "fairly".
*   *Tip:* This can be useful to avoid forcing special "final output flows" from edge proportionality, such as waste and losses. It is also useful to prioritize domestic sectors over exports.

#### 2. Behavior control flags
These boolean flags determine the "physics" of your simulation:

*   **`fix_primary_diagonals`** (bool, Default: `True`):
    *   `True`: **Rigid supply.** The production capacity of primary sectors is locked. The model must satisfy new demand solely by reallocating existing resources.
    *   `False`: **Flexible supply.** Primary production can scale up or down to meet demand.
*   **`allow_neg_primary` / `_intermediate` / `_output`** (bool, Default: `False`):
    *   By default, row sums (net output) must be non-negative. Setting this to `True` allows a row's sum to become negative, representing an unrealistic theoretical deficit.

#### 3. Penalty weights and fine-tuning
The model minimizes a weighted cost function. Adjusting these weights tells the solver which outcomes are "expensive" and should be avoided.

*   **`deficit_weight_*`** (float): penalty for creating a **deficit** (shortfall) in primary (Default: `0`), intermediate (Default: `1.0`), or output (Default: `1.0`) rows. Higher weights makes the model fight harder to avoid deficits in those sectors.
*   **`surplus_weight_*`** (float): penalty for creating a **surplus** (overproduction) in primary (Default: `1.0`), intermediate_direct (Default: `1.0`), intermediate_indirect (Default: `1.0`), or output (Default: `1.0`) rows. Higher weights makes the model fight harder to avoid surpluses in those sectors.
*   **`slack_weight`** (float, Default: `1e6`): **Critical parameter.** This penalizes failure to meet the requested `new_demands`.
    *   *Tip:* if the model fails to reach your target demand, increase this to `1e9` to force strict adherence.
*   **`max_deficit_penalty`** (float, Default: `0`): penalizes the single largest deficit in the system. Increasing this encourages "fairness," spreading small deficits across many sectors rather than allowing one sector to crash completely.
*   **`tol`** (float, Default: `1e-8`): the numerical feasibility tolerance for the solver. It defines the precision with which the solver satisfies constraints. You typically only need to adjust this if you encounter numerical stability issues with very large or complex matrices.
*   *Tip:* if the model finds no feasible solution try to increase this to reach a solution at the expense of precision.

### Example usage

```python
# 1. Setup inputs
M_path = 'data/bioeconomy_matrix.csv'
targets = [15]        # Row indexes for targeted product
demands = [500.0]  # New target demand values
export_rows = [10,11,14]  # Row indexes for exported products with low priority

# 2. Run the simulation
# Scenario: rigid primary supply, but strictly enforcing the new demand targets
adjusted_matrix, results = solve_adjustable_model(
    M_csv_path=M_path,
    row_class=row_classification, # Defined elsewhere
    row_ratios=r_ratios, # Defined elsewhere
    column_ratios=c_ratios, # Defined elsewhere
    target_rows=targets,
    new_demands=demands,
    
    # --- Configuration ---
    fix_primary_diagonals=True,       # Cannot grow primary extraction
    slack_weight=1e9,                 # Must meet the 500 & 120 targets
    deficit_weight_intermediate=10.0, # Avoid deficits in intermediate steps
    distribution='row_sum',           # Enforce edge proportionality
    special_rows = export_rows        # Prioritize domestic sectors over foreign demand
    
)

# 3. Check results
if adjusted_matrix is not None:
    print("Optimization successful.")
    print(results.head())
```

### Interpreting the output

The function returns two objects:

1.  **`adjusted_matrix`** (DataFrame): the fully calculated matrix representing the new bioeconomy state. Returns `None` if the solver finds the constraints physically impossible to satisfy.
2.  **`results_df`** (DataFrame): a diagnostic table detailing the shift for every row:
    *   **Original / final row sum**: the total flow before and after the simulation.
    *   **Delta**: the net change (`final - original`).
    *   **Deficit**: the amount by which supply fell short of requirements.
    *   **Surplus**: the amount of excess production generated (often due to rigid co-production ratios).


In [ ]:
# Specify the classification for each row/column in the matrix.

# Define the three string values for the dictionary
class1 = "primary"
class2 = "intermediate"
class3 = "output"

# Define the keys that will receive each string value, i.e., the row index
# Primary
keys_for_class1 = list(range(24)) # We have sorted the matrix and we know that the first 25 rows are primary otherwise = [0,1...24]
# Output
keys_for_class3 = [26,28,30,37,39,40,43,44,60,68,74,78,79,83,87,89,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107]

# Automatically determine the upper limit for the range (max of keys_for_class3)
upper_limit = max(keys_for_class3)

# Get all keys from 0 to the upper limit
all_keys = list(range(0, upper_limit + 1))

# Build the dictionary
row_class = {}

# Assign the values to the respective keys
for key in keys_for_class1:
    row_class[key] = class1

for key in keys_for_class3:
    row_class[key] = class3

# Assign values to the remaining keys, those not assigned yet, i.e., intermediates
for key in all_keys:
    if key not in row_class:
        row_class[key] = class2


In [ ]:
# Now specify which columns are bound by technical coefficient ratios.
tech_coeff = {
    0: [27,87],
     2: [28, 93, 87],
     3: [35,36],
     4: [29, 70],
     5: [41,42],
     7: [49,50],
     9: [52,53,54],          
     10: [61,62],
     11: [63,64],
     12: [65,66],
     14: [72,73],
     15: [75,76],
     16: [29, 70],
     18: [44,87],
     20: [29, 70],
     26: [26,87],
     28: [28,87],
     35: [35,40,69,83],     
     36: [36,40,60,74,83],   
     40: [25,40,87],     
     41: [41,43,83],    
     42: [42,60],     
     44: [44,45],
     45: [0,97,40,45],   
     46: [40,46,83],
     49: [49,69,83],
     50: [40,50,60,74,83],
     51: [40,43,51,83],     
     52: [52,87],
     53: [53,69,83],   
     54: [43,54,83],      
     58: [57,58,59],
     61: [40,61,69,83],                 
     62: [40,60,62,74,83],
     63: [63,69,83],
     64: [40,60,64,74,83],
     65: [43,65,83],
     66: [40,60,66,74,83],
     68: [68,87],
     69: [69,87],
     72: [40,69,72,83],
     73: [60,73],
     75: [40,69,75,83],    
     76: [40,60,74,76,83],
     81: [79,81],
     85: [69,83,85]
}

# Add empty key for rows that have no technical ratios 
original_row_ratios = {key: [] for key in row_class.keys()}
original_row_ratios.update(tech_coeff)

#Initialize also bill of materials - column ratios - if necessary. Empty by default
column_ratios = {}

### Classification summary for outputs:

    26: Animal-based food
    28: Aquatic-based food
    30: Bioenergy
    37: Energy
    43: Fibres and others
    44: Fishmeal & oil
    60: Not harvested residues
    68: Plant-based food
    74: Residues - unknown use
    78: Solid wood products
    79: Solid wood products exports
    83: Unknown/Losses
    89: Wood pulp
    93: Export of Capture fisheries
    94: Export of Animal products (feed eq.)
    95: Export of Cereals
    96: Export of Fibre Crops
    97: Export of Fishmeal & oil products
    98: Export of Fodder crops
    99: Export of Fruits
    100: Export of Grazing products
    101: Export of Imported Processed products (biomass eq.)
    102: Export of Oil crops
    103: Export of Olive trees
    104: Export of Other industrial crops
    105: Export of Pulses & protein crops
    106: Export of Root crops
    107: Export of Vegetables


### Some simple examples

Let's have a try with considering the food-feed-energy competition.
To ensure that this substitution in feasible we need to ensure that their common inputs (e.g., grains) can be redirected from one to the others.
To do so, we remove the columns of "Feed & bedding" (livestock farming) and "Plant-based food supply" from all row ratios that contain them.

In [ ]:
# First we make a copy of the original ratios, to be safe
food_feed_row_ratios = original_row_ratios
# Then we clean both row ratios from the copy we just made
# Row 60 = "Plant-based food supply": allow redirect inputs to other supply chains
for key in food_feed_row_ratios:
    if key != 69:
        food_feed_row_ratios[key] = [x for x in food_feed_row_ratios[key] if x != 69]

#Row 40 = "Feed & bedding": allows receiving inputs from other supply chains
for key in food_feed_row_ratios:
    if key != 40:
        food_feed_row_ratios[key] = [x for x in food_feed_row_ratios[key] if x != 40]


In [ ]:
## The potentially avaialble supply without a "food first" policy implemented
## This means that both plant-based and animal-based food can be affected

target_rows = [30] #Bioenergy (no-wood)
new_demands = [20510] #Initial value = 12432

exclude_surplus_rows = []
exclude_deficit_rows = []# Allows grain/residue substitution in biofuels
special_rows = [40,60,83,87,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107]   
#40: Feed & bedding
#60: Not harvested residues
#83: Unknown/Losses
#87: Waste
#93-107: exports

M_csv_path = "start_matrix_with_exp.csv"

adj, defs = solve_adjustable_model(
    M_csv_path=M_csv_path,
    row_class=row_class,
    target_rows=target_rows,
    new_demands=new_demands,
    row_ratios=food_feed_row_ratios, ####Check
    column_ratios=column_ratios,
    slack_rows=list(range(len(row_class))),
    exclude_surplus_rows=exclude_surplus_rows,
    exclude_deficit_rows=exclude_deficit_rows,
    special_rows =special_rows 
)

In [ ]:
# Now we do impose a food-first policy with:
# There cannot be a deficit of food

target_rows = [30] #Bioenergy (no-wood)
new_demands = [12513] #Initial value = 12432 MAX with row_sum = 12513 (depletion of exports (grains))

## The potentially avaialble supply is very limited!
## Running it with higher new_demands value higher than 12513 will cause infeasibility 

exclude_surplus_rows = []
exclude_deficit_rows = [26,28,68,60]   # Ensures no grain/residue substitution in biofuels is allowed PLUS no use of further non-harvested residues is allowed
special_rows = [40,60,83,87,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107]   
#40: Feed & bedding
#60: Not harvested residues
#83: Unknown/Losses
#87: Waste
#93-107: exports

M_csv_path = "start_matrix_with_exp.csv"

adj, defs = solve_adjustable_model( # adj is the adjusted matrix and defs is the summary of the changes
    M_csv_path=M_csv_path,
    row_class=row_class,
    target_rows=target_rows,
    new_demands=new_demands,
    row_ratios=food_feed_row_ratios,
    column_ratios=column_ratios,
    slack_rows=list(range(len(row_class))),
    exclude_surplus_rows=exclude_surplus_rows,
    exclude_deficit_rows=exclude_deficit_rows,
    special_rows =special_rows 
)

In [ ]:
# Here we look at aquatic-based food, which are subject to much less competition

target_rows = [28] #Aquatic-based food
new_demands = [4100] #Initial value = 4050

exclude_surplus_rows = []
exclude_deficit_rows = []
special_rows = [40,60,83,87,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107]   
#40: Feed & bedding
#60: Not harvested residues
#83: Unknown/Losses
#87: Waste
#93-107: exports

M_csv_path = "start_matrix_with_exp.csv"

adj, defs = solve_adjustable_model(
    M_csv_path=M_csv_path,
    row_class=row_class,
    target_rows=target_rows,
    new_demands=new_demands,
    row_ratios=original_row_ratios, 
    column_ratios=column_ratios,
    slack_rows=list(range(len(row_class))),
    exclude_surplus_rows=exclude_surplus_rows,
    exclude_deficit_rows=exclude_deficit_rows,
    special_rows =special_rows 
)

The problem is infeasible, meaning that there is no way to increase the supply of aquatic-based food, not even by redirecting biomass flows from other sources

In [ ]:
# A similar example can come from fibers and other. This supply come from very few specific sources among which there is no competition

target_rows = [43] #Fibres and others
new_demands = [1453] #Initial value = 1353

exclude_surplus_rows = []
exclude_deficit_rows = []
special_rows = [40,60,83,87,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107]   
#40: Feed & bedding
#60: Not harvested residues
#83: Unknown/Losses
#87: Waste
#93-107: exports

M_csv_path = "start_matrix_with_exp.csv"

adj, defs = solve_adjustable_model(
    M_csv_path=M_csv_path,
    row_class=row_class,
    target_rows=target_rows,
    new_demands=new_demands,
    row_ratios=original_row_ratios, ####Check
    column_ratios=column_ratios,
    slack_rows=list(range(len(row_class))),
    exclude_surplus_rows=exclude_surplus_rows,
    exclude_deficit_rows=exclude_deficit_rows,
    special_rows =special_rows 
)

The problem is infeasible even in this case. This means that there is no way to increase the supply of fibres and other, not even by redirecting biomass flows from other sources

In [ ]:
#Let's look at wood-based energy products now

target_rows = [37] #Energy - wood-based
new_demands = [200000] #Initial value = 194851
special_rows = [40,60,83,87,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107]   
#40: Feed & bedding
#60: Not harvested residues
#83: Unknown/Losses
#87: Waste
#93-107: exports

exclude_surplus_rows = []
exclude_deficit_rows = []

M_csv_path = "start_matrix_with_exp.csv"

adj, defs = solve_adjustable_model(
    M_csv_path=M_csv_path,
    row_class=row_class,
    target_rows=target_rows,
    new_demands=new_demands,
    row_ratios=original_row_ratios,
    column_ratios=column_ratios,
    slack_rows=list(range(len(row_class))),
    exclude_surplus_rows=exclude_surplus_rows,
    exclude_deficit_rows=exclude_deficit_rows,
    special_rows =special_rows 
)

In [ ]:
# What happens instead if we demand more solid wood products?

target_rows = [78] #Solid wood products
new_demands = [70000] #Initial value = 65505

exclude_surplus_rows = []
exclude_deficit_rows = []
special_rows = [40,60,83,87,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107]   
#40: Feed & bedding
#60: Not harvested residues
#83: Unknown/Losses
#87: Waste
#93-107: exports

M_csv_path = "start_matrix_with_exp.csv"

adj, defs = solve_adjustable_model(
    M_csv_path=M_csv_path,
    row_class=row_class,
    target_rows=target_rows,
    new_demands=new_demands,
    row_ratios=original_row_ratios, ####Check
    column_ratios=column_ratios,
    slack_rows=list(range(len(row_class))),
    exclude_surplus_rows=exclude_surplus_rows,
    exclude_deficit_rows=exclude_deficit_rows,
    special_rows =special_rows 
)

## Expanding the matrix to analyze the competition due to bioeconomy expansion

### Core Concept

The bioeconomy is expected to expand and develop with new biobased products being demanded.
To show these new products and investigate the effect of their demand, the bioeconomy matrix must first be expanded.

The expansion consists in:
#### Augmenting the matrix
A new row and column is added to represent the new biobased product

#### Linking
The new biobased product is linked to the existing bioeconomy via the proper nodes, i.e., reflecting the feedstock that can be used

#### Remapping the dictionaries
The dictionaries need to be updated to reflect the additional product

### The consequences of demanding more new biobased product are calculated 


In [ ]:
# Add a new process that gets input from domestic roundwood: observe the effects of 
# competition on existing products relying on domestic roundwood (or materials linked downstream)

# Set the starting Dataframe path
M_csv_path = "start_matrix_with_exp.csv"

# Read the initial DataFrame
M_df_AA = pd.read_csv(M_csv_path, header=0, index_col=0).fillna(0).apply(pd.to_numeric)

# Check initial matrix size
print(M_df_AA.shape)
# Step 1: Add a new column and row "Adipic acid", initialized with float values
M_df_AA["Adipic acid"] = 0.0
M_df_AA.loc["Adipic acid"] = 0.0

# Step 2: Assign a small negative value to rows in the "Adipic acid" column where the index contains "Domestic roundwood". This signals the linkage
M_df_AA.loc[M_df_AA.index.str.contains("Domestic roundwood", case=False), "Adipic acid"] = -0.0001
# Step 3: Assign the same value to the corresponding diagonal
M_df_AA.loc["Adipic acid", "Adipic acid"] = 0.0001

# Check final matrix size
print(M_df_AA.shape)

In [ ]:
# Make a copy of the original ratios
residues_row_ratios = original_row_ratios
residues_column_ratios = column_ratios
residues_row_class = row_class

# Adipic acid was added at the edge of the matrix, so its index corresponds to the matrix size minus 1 (index starts at 0)
num = len(M_df_AA)-1

# Add the new Adipic acid process the the copied dictionaries. It is an output
residues_row_ratios[num] = []     
residues_column_ratios[num] = []  
residues_row_class[num] = "output"
len(M_df_AA)

# Now define the parameters
target_rows = [108] # AA linked to domestic roundwood
new_demands = [2389.44] # Demand for forestry residues to cover EU 2020 AA production
exclude_surplus_rows = []
exclude_deficit_rows = []
special_rows = [40,60,83,87,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107]   
#40: Feed & bedding
#60: Not harvested residues
#83: Unknown/Losses
#87: Waste
#93-107: exports

M_csv_path = M_df_AA

# And finally call the function
adj, defs = solve_adjustable_model(
    M_csv_path=M_csv_path,
    row_class=residues_row_class,
    target_rows=target_rows,
    new_demands=new_demands,
    row_ratios=residues_row_ratios, 
    column_ratios=residues_column_ratios,
    slack_rows=list(range(len(row_class))),
    exclude_surplus_rows=exclude_surplus_rows,
    exclude_deficit_rows=exclude_deficit_rows,
    special_rows =special_rows 
)